# Automatic Program Generation for the Steane Code

GuppyFT is an extension of the Guppy quantum programming language to aid writing, compiling and running fault-tolerant programs by providing a separation of concerns between writing quantum algorithms and quantum error correction (QEC). Researchers are able to write QEC-agnostic programs, and utilise QEC code architectures developed by experts to automatically encode into a fault-tolerant program.

This notebook provides an introduction to the encoding features of GuppyFT through simple examples.

## Encoding a Guppy program

Encoding happens in four stages:
1. Write a computational Guppy program using `guppylang.std.quantum` operations.
2. Run an encoding pass to convert the computational program into a logical program.
3. (Optional) Run logical-aware optimisation passes.
4. Replace the logical operations with physical implementations.

Let's see a full workflow with a quantum teleportation program.

In [1]:
from guppylang import guppy
from guppylang.std import quantum as qlib
from guppylang.std.builtins import array, owned
from guppylang.std.platform import output

from hugr.hugr.base import Hugr

First, we write our computational program. At this stage, we are not concerned with QEC. The program is written assuming that it will run on a noiseless quantum computer using `guppylang.std` operations.

In [2]:
@guppy
def teleport_primitive(src: qlib.qubit @ owned) -> qlib.qubit:
    # Create Bell pair
    tmp = qlib.qubit()
    tgt = qlib.qubit()
    qlib.h(tmp)
    qlib.cx(tmp, tgt)

    # Teleport
    qlib.cx(src, tmp)
    qlib.h(src)
    if qlib.measure(src).read():
        qlib.z(tgt)
    if qlib.measure(tmp).read():
        qlib.x(tgt)

    return tgt


@guppy
def teleport() -> None:
    # Qubit to be teleported
    src = qlib.qubit()

    # Teleport `src` to `tgt`
    tgt = teleport_primitive(src)
    output("tgt", qlib.measure(tgt).read())

Now we can define the QEC code architecture that we would use to encode our program to be fault-tolerant. An architecture includes the necessary encoding and implementation passes to convert our computational program into a runnable, physical package. This includes:
- An encoding pass to convert from computational to logical.
- (Optional) Optimisations on the logical program.
- Physical implementations of the logical gadgets.

For this example, we will use the Steane architecture.

In [3]:
from guppyft.code.steane.encode import SteaneBuilder

# Define a Steane architecture instance.
# The only required parameter is an upper bound
# to the total number of logical blocks available
# during program execution.

steane = SteaneBuilder().build(n_blocks=3)

We can use our architecture to encode the program, running the full pass from computational to logical and finally to physical, producing a package that we can run.

In [4]:
teleport_unencoded = teleport.compile()
teleport_encoded = steane.encode(teleport_unencoded)

As both the physical and computational program are represented by HUGR graphs, we can count the number of nodes to get an idea of the size of the two programs. We expect the physical program to be much bigger as it will contain the physical implementation of everything we do with the three computational qubits. 

First we'll look at the computational program for teleportation.

In [5]:
computational_hugr: Hugr = teleport_unencoded.modules[0]


# Helper function to count the number of Hugr nodes of a particular type
def _count_ops(hugr: Hugr, string_name: str) -> int:
    count = 0
    for _, data in hugr.nodes():
        if string_name in data.op.name():
            count += 1

    return count


print("Total number of nodes before encoding:", computational_hugr.num_nodes())
print(
    "Hadamard count before encoding:", _count_ops(computational_hugr, "tket.quantum.H")
)
print("CX count before encoding:", _count_ops(computational_hugr, "tket.quantum.CX"))

Total number of nodes before encoding: 48
Hadamard count before encoding: 2
CX count before encoding: 2


Now let's compute the same counts for the much larger physical program.

In [6]:
physical_hugr: Hugr = teleport_encoded.modules[0]

print("Total number of nodes after encoding:", physical_hugr.num_nodes())
print("Hadamard count after encoding:", _count_ops(physical_hugr, "tket.quantum.H"))
print("CX count after encoding:", _count_ops(physical_hugr, "tket.quantum.CX"))

Total number of nodes after encoding: 7859
Hadamard count after encoding: 13
CX count after encoding: 44


# Dynamic allocation of logical qubits

GuppyFT provides support for arbitrary control flow through dynamic allocation of logical qubits. Below is an example of qubits being dynamically allocated based on the outcome of a qubit measurement.

In [7]:
@guppy
def dynamic_allocation() -> None:
    q0 = qlib.qubit()
    qlib.h(q0)
    if qlib.measure(q0):  # If true, allocate a single qubit
        q_arr = array(qlib.qubit())
        for q in q_arr:
            output("q", qlib.measure(q).read())
    else:  # Otherwise, allocate two qubits
        q_arr = array(qlib.qubit(), qlib.qubit())
        qlib.cx(q_arr[0], q_arr[1])
        for q in q_arr:
            output("q", qlib.measure(q).read())


dyn_alloc_encoded = (
    SteaneBuilder().build(n_blocks=3).encode(dynamic_allocation.compile()).to_bytes()
)

# Mid-circuit measurement and qubit reuse

The Quantinuum stack supports mid-circuit measurements and qubit reuse. This support continues at the logical level with GuppyFT.

We can demonstrate this using the `teleport_primitive` defined above to teleport a qubit twice. Without qubit reuse, this program would require 5 qubits. However, only 3 qubits are required at any one time.

In [8]:
@guppy
def qubit_reuse() -> None:
    # Qubit to teleport
    src = qlib.qubit()

    # Run teleportation twice
    tgt = teleport_primitive(src)
    src = teleport_primitive(tgt)

    output("src", qlib.measure(src).read())


# Encode with only a single logical block to demonstrate logical reuse.
reuse_encoded = SteaneBuilder().build(n_blocks=3).encode(qubit_reuse.compile())

# Dynamic QEC cycle injection

In quantum programs with complex control flow, it can be useful to use information obtained during runtime to determine when to inject a QEC cycle. GuppyFT supports dynamic injection of QEC cycles depending on the logical gates that have been performed.

In this demonstration, each logical operation is assigned a cost, which is tracked on a per-block basis. At runtime, once a threshold is reached, a QEC cycle is injected, and the tracking counter is reset to 0.

In [9]:
from guppyft.code.steane.encode import QECPolicy, QECStyle

# Define a QEC policy using Steane style syndrome extraction
# We set the threshold to be 2.
# Both the logical `H` and `CX` gates each have a cost of 1.
qec_policy = QECPolicy(style=QECStyle.Steane, threshold=2)
qec_policy.set_cost("H", 1.0)
qec_policy.set_cost("CX", 1.0)

# We can now provide the `qec_policy` to define our Steane architecture.
steane_qec = SteaneBuilder().with_qec_policy(qec_policy).build(n_blocks=2)

In [10]:
# Demonstration program to apply `H` and `CX` gates to two qubits.
@guppy
def qec_cycles() -> None:
    q0 = qlib.qubit()
    q1 = qlib.qubit()

    # Track costs:      [q0 , q1 ]
    qlib.h(q0)  #       [1.0, 0.0]
    qlib.cx(q0, q1)  #  [2.0, 1.0]
    #                   [0.0, 1.0] <-- QEC cycle on q0
    qlib.h(q1)  #       [1.0, 2.0]
    #                   [1.0, 0.0] <-- QEC cycle on q1

    output("q0", qlib.measure(q0).read())
    output("q1", qlib.measure(q1).read())


# Encode two programs with, and without our QEC policy.
no_qec_cycles_encoded = (
    SteaneBuilder().build(n_blocks=2).encode(qec_cycles.compile()).to_bytes()
)
qec_cycles_encoded = (
    SteaneBuilder()
    .with_qec_policy(qec_policy)
    .build(n_blocks=2)
    .encode(qec_cycles.compile())
    .to_bytes()
)

# State factories

For the Steane zero-state preparation, we are using a repeat-until-success scheme that measures a flag ancilla to verify that the state preparation succeeds. The runtime pauses after each preparation to measure the ancilla qubit; this implies that state preparations are performed sequentially.

Here, we demonstrate using state factories in GuppyFT to prepare multiple states in parallel, pulling from the factory when new states are required.

In [11]:
from guppyft.code.steane.encode import RUSStateFactoryConf

# Define the state factory configuration.
# Our factory will prepare 2 states in parallel,
# and make 5 RUS attempts for each state.
factory_conf = RUSStateFactoryConf(size=2, max_attempts=5)

# Use the factory configuration to define our Steane architecture.
steane_code_factories = (
    SteaneBuilder()
    .with_qec_policy(qec_policy)
    .with_zero_factory_conf(factory_conf)
    .build(n_blocks=2)
)
qec_factory_encoded = steane_code_factories.encode(qec_cycles.compile()).to_bytes()